In [1]:
import pandas as pd
import random
import joblib
from sklearn.ensemble import RandomForestRegressor

In [2]:


# 1. Setup Rules
PROJECT_MAPPING = {"AI_Model": 0, "Web_App": 1, "Mobile_App": 2, "Research": 3}
PROJECT_CONFIG = {
    0: {"coding": 0.6, "mgmt": 0.1, "design": 0.1, "math": 0.2},
    1: {"coding": 0.4, "mgmt": 0.2, "design": 0.4, "math": 0.0},
    2: {"coding": 0.5, "mgmt": 0.2, "design": 0.3, "math": 0.0},
    3: {"coding": 0.2, "mgmt": 0.1, "design": 0.0, "math": 0.7},
}

# PROFESSIONAL + EXPERIENCED - 9pts
# PROFESSIONAL - 7pts
# FRESHER - 5pts
# AMATEUR - 3pts

data = []

# 2. Generate Data
for _ in range(7000):
    p_type_code = random.choice(list(PROJECT_MAPPING.values()))
    team_size = random.randint(3, 9)

    # 10% of the rows to be "Perfect"
    if random.random() < 0.10:
        avg_skill = random.uniform(9, 10)
    else:
        avg_skill = random.uniform(1, 8)

    # Generate Skills based on that average
    stats = {
        "coding": int(avg_skill * team_size),
        "mgmt":   int(avg_skill * team_size * random.uniform(0.5, 1.0)),
        "design": int(avg_skill * team_size),
        "math":   int(avg_skill * team_size)
    }

    # Calculate "Ground Truth" Success
    weights = PROJECT_CONFIG[p_type_code]
    cap = team_size * 9

    weighted_score = 0
    max_possible = 0

    for k, w in weights.items():
        val = min(stats[k], cap)
        weighted_score += val * w
        max_possible += cap * w

    penalty = (team_size - 6) * 0.02 if team_size > 6 else 0

    base_prob = (weighted_score / max_possible) - penalty
    success = max(0, min(1, base_prob))

    data.append([p_type_code, team_size, stats['coding'], stats['mgmt'], stats['design'], stats['math'], success])

# 3. train Model
df = pd.DataFrame(data, columns=['Project_Type', 'Team_Size', 'Skill_Coding', 'Skill_Mgmt', 'Skill_Design', 'Skill_Math', 'Success_Rate'])
print(df)


      Project_Type  Team_Size  Skill_Coding  Skill_Mgmt  Skill_Design  \
0                0          8             9           6             9   
1                1          8            14          13            14   
2                1          8            13          12            13   
3                2          9            65          44            65   
4                3          4             6           4             6   
...            ...        ...           ...         ...           ...   
6995             1          8            21          11            21   
6996             3          4            21          18            21   
6997             0          9            54          39            54   
6998             3          8            22          18            22   
6999             2          9            87          78            87   

      Skill_Math  Success_Rate  
0              9      0.080833  
1             14      0.151667  
2             13      0.

In [3]:
X = df.drop('Success_Rate', axis=1)
y = df['Success_Rate']

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# 4. Save & Test
joblib.dump(model, "project_success_brain.pkl")

print("RETRAINING COMPLETE")

# Test your "Dream Team" again immediately
test_team = [[0, 5, 44, 44, 44, 44]]
pred = model.predict(test_team)[0]
print(f"New Prediction for Dream Team: {pred * 100:.2f}%")

RETRAINING COMPLETE
New Prediction for Dream Team: 89.12%


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [4]:
model = joblib.load("project_success_brain.pkl")

# 2. Define a Mapping (Must match training step)
PROJECT_MAPPING = {"AI_Model": 0, "Web_App": 1, "Mobile_App": 2, "Research": 3}

def ai_predict_success(project_name, team_stats):
    # Convert input to the format the AI expects
    # Expected Order: [Project_Type, Team_Size, Coding, Mgmt, Design, Math]

    p_code = PROJECT_MAPPING[project_name]

    features = pd.DataFrame([[
        p_code,
        team_stats['size'],
        team_stats['coding'],
        team_stats['mgmt'],
        team_stats['design'],
        team_stats['math']
    ]], columns=['Project_Type', 'Team_Size', 'Skill_Coding', 'Skill_Mgmt', 'Skill_Design', 'Skill_Math'])

    prediction = model.predict(features)[0]
    return round(prediction, 2)

# Model testing
new_team = {
    'size': 5,
    'coding': 45,
    'mgmt': 44,
    'design': 44,
    'math': 45
}

# Predict for AI Project
score_ai = ai_predict_success("AI_Model", new_team)
print(f"AI Project Success Probability: {score_ai * 100:.2f}%")

# Predict for Web App
score_web = ai_predict_success("Web_App", new_team)
print(f"Web App Success Probability: {score_web * 100:.2f}%")

AI Project Success Probability: 96.00%
Web App Success Probability: 96.00%


In [5]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import os

#CONFIG MUST BE FIRST
st.set_page_config(page_title="AI Project Predictor", page_icon="🚀")

st.title(" AI Team Success Predictor")

# FILE LOADING
current_dir = os.path.dirname(os.path.abspath(__file__))
model_path = os.path.join(current_dir, "project_success_brain.pkl")

try:
    model = joblib.load(model_path)
    st.success(f"Brain loaded successfully from: {current_dir}") # Debugging
except FileNotFoundError:
    st.error("Model Not Found!")
    st.code(f"Looking for file at:\n{model_path}")
    st.warning("Make sure 'project_success_brain.pkl' is in the SAME folder as 'app.py'")
    st.stop()

# 3. INPUTS 
st.sidebar.header("Configuration")
project_type = st.sidebar.selectbox("Project Type", ("AI_Model", "Web_App", "Mobile_App", "Research"))
team_size = st.sidebar.slider("Team Size", 3, 10, 5)

max_val = team_size * 10
coding = st.sidebar.slider("Total Coding Skill", 0, max_val, int(max_val * 0.5))
mgmt = st.sidebar.slider("Total Management Skill", 0, max_val, int(max_val * 0.3))
design = st.sidebar.slider("Total Design Skill", 0, max_val, int(max_val * 0.3))
math = st.sidebar.slider("Total Math Skill", 0, max_val, int(max_val * 0.3))

# 4. PREDICT 
if st.button("Predict Success"):
    p_map = {"AI_Model": 0, "Web_App": 1, "Mobile_App": 2, "Research": 3}
    input_data = pd.DataFrame([[p_map[project_type], team_size, coding, mgmt, design, math]], 
                              columns=['Project_Type', 'Team_Size', 'Skill_Coding', 'Skill_Mgmt', 'Skill_Design', 'Skill_Math'])
    
    prediction = model.predict(input_data)[0] * 100
    
    st.divider()
    st.metric("Probability", f"{prediction:.2f}%")
    st.progress(int(prediction))


Overwriting app.py


In [6]:
import pandas as pd
import random
import joblib
from sklearn.metrics import mean_absolute_error, r2_score

print("GENERATING 2,000 TEST SAMPLES")

# 1. Define Rules (Must match training logic)
PROJECT_MAPPING = {"AI_Model": 0, "Web_App": 1, "Mobile_App": 2, "Research": 3}
PROJECT_CONFIG = {
    0: {"coding": 0.6, "mgmt": 0.1, "design": 0.1, "math": 0.2},
    1: {"coding": 0.4, "mgmt": 0.2, "design": 0.4, "math": 0.0},
    2: {"coding": 0.5, "mgmt": 0.2, "design": 0.3, "math": 0.0},
    3: {"coding": 0.2, "mgmt": 0.1, "design": 0.0, "math": 0.7},
}

test_data = []

# 2. Generate 2000 Rows
for _ in range(2000):
    p_type_code = random.choice(list(PROJECT_MAPPING.values()))
    team_size = random.randint(3, 9)
    
    # Mix of "Perfect" and "Normal" teams
    if random.random() < 0.15:
        avg_skill = random.uniform(9, 10)
    else:
        avg_skill = random.uniform(1, 8.5)
        
    stats = {
        "coding": int(avg_skill * team_size),
        "mgmt":   int(avg_skill * team_size * random.uniform(0.5, 1.0)), 
        "design": int(avg_skill * team_size),
        "math":   int(avg_skill * team_size)
    }
    
    # Calculate Ground Truth
    weights = PROJECT_CONFIG[p_type_code]
    cap = team_size * 9
    weighted_score = sum(min(stats[k], cap) * w for k, w in weights.items())
    max_possible = sum(cap * w for k, w in weights.items())
    penalty = (team_size - 6) * 0.02 if team_size > 6 else 0
    
    success = max(0, min(1, (weighted_score / max_possible) - penalty))
    
    test_data.append([p_type_code, team_size, stats['coding'], stats['mgmt'], stats['design'], stats['math'], success])

# 3. Create DataFrame
df_test = pd.DataFrame(test_data, columns=['Project_Type', 'Team_Size', 'Skill_Coding', 'Skill_Mgmt', 'Skill_Design', 'Skill_Math', 'Success_Rate'])

# Save to CSV 
df_test.to_csv("testing_data.csv", index=False)
print("'testing_data.csv' created with 2,000 rows.")

# VALIDATION STEP

# 4. Load the AI Model
try:
    model = joblib.load("project_success_brain.pkl")
    print("Model loaded successfully.")
except:
    print("Error: Model not found. Run the Training step first!")
    exit()

# 5. Predict New Data
X_test = df_test.drop('Success_Rate', axis=1)
y_true = df_test['Success_Rate']

print("Asking AI to predict these 2,000 outcomes...")
y_pred = model.predict(X_test)

# 6. Check Accuracy
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print("-" * 30)
print(f"Mean Absolute Error: {mae:.4f}")
print(f"Accuracy Score : {r2:.4f} / 1.0")
print("-" * 30)

# Show a few examples
df_test['AI_Prediction'] = y_pred
print("\n Comparisons (Actual vs AI):")
print(df_test[['Success_Rate', 'AI_Prediction']].to_string(index=False))

GENERATING 2,000 TEST SAMPLES
'testing_data.csv' created with 2,000 rows.
Model loaded successfully.
Asking AI to predict these 2,000 outcomes...
------------------------------
Mean Absolute Error: 0.0049
Accuracy Score : 0.9987 / 1.0
------------------------------

 Comparisons (Actual vs AI):
 Success_Rate  AI_Prediction
     0.698025       0.704284
     0.255833       0.253972
     0.175926       0.174833
     0.912840       0.914407
     0.587778       0.601014
     0.330833       0.329111
     0.097500       0.094500
     0.351852       0.336889
     0.682222       0.680289
     0.368889       0.370756
     0.451111       0.450311
     0.533333       0.531093
     0.300000       0.306259
     0.878272       0.877951
     0.769444       0.763556
     0.851667       0.887194
     0.294722       0.293542
     0.342469       0.338506
     0.244444       0.241000
     0.768333       0.769389
     0.896111       0.898898
     1.000000       1.000000
     0.577778       0.579911
     0.5